# Notebook 3 — Baseline Models

**Goal:** Establish the *floor* — what can dumb, simple models achieve? Only if our ML model beats these baselines is it worth the added complexity.

## Why baselines matter

Imagine you build an XGBoost model that predicts train delay with a Mean Absolute Error of 3.2 minutes. Is that good?

You have no idea — until you compare it to a baseline. If a baseline of "always predict 0 minutes delay" achieves MAE = 3.5 minutes, your XGBoost barely helped. If the baseline achieves MAE = 7.0 minutes, your model is excellent.

**Rule:** Never build a complex model until you've measured a dumb one.

## Our baselines

1. **Zero baseline** — predict delay = 0 for every train. ("I don't know, assume on time.")
2. **CTA's own prediction** — use `arr_t - scheduled_time` directly. This is what already exists.
3. **Historical median** — for each (route, station, hour, weekday/weekend) bucket, predict the median delay from historical data.
4. **Ridge regression** — a linear model; the simplest ML approach.

## Metrics we'll use

For **regression** (predicting delay_minutes):
- **MAE** (Mean Absolute Error): average |predicted - actual|. Easy to interpret — "on average off by X minutes."
- **RMSE** (Root Mean Squared Error): penalizes large errors more. Useful when big misses are especially costly.

For **classification** (ahead / on_time / behind):
- **Accuracy**: what fraction did we label correctly?
- **Macro F1**: average F1 across all 3 classes. Better than accuracy when classes are imbalanced.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, f1_score, confusion_matrix
)

sns.set_theme(style='darkgrid')

DATA_PATH = Path('../data/features.parquet')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run notebook 02 first to generate data/features.parquet')

df = pd.read_parquet(DATA_PATH)
print(f'Loaded {len(df):,} labelled rows')
df.head(2)

## The critical rule: time-based train/test split

**Never use random train/test splits for time-series data.** Here's why:

If you randomly split, your test set might contain rows from January 10th, but your training set contains rows from January 11th. Your model has effectively "seen the future" — it's been trained on data that comes after what it's supposed to predict.

This produces overly optimistic evaluation metrics. In production, the model would perform worse.

**Correct approach:** train on all data up to date T, test on data after T. This mimics real-world deployment.

In [ ]:
df = df.sort_values('snapshot_time')

# Use the last 2 weeks as test set
cutoff = df['snapshot_time'].max() - pd.Timedelta(days=14)
train = df[df['snapshot_time'] <= cutoff].copy()
test  = df[df['snapshot_time'] >  cutoff].copy()

print(f'Train: {len(train):,} rows  ({train.snapshot_time.min().date()} → {train.snapshot_time.max().date()})')
print(f'Test:  {len(test):,}  rows  ({test.snapshot_time.min().date()} → {test.snapshot_time.max().date()})')

fig, ax = plt.subplots(figsize=(12, 2))
ax.barh(0, len(train), color='steelblue', label=f'Train ({len(train):,})')
ax.barh(0, len(test), left=len(train), color='tomato', label=f'Test ({len(test):,})')
ax.set_yticks([])
ax.set_xlabel('rows (time order →)')
ax.legend()
ax.set_title('Time-based train/test split')
plt.tight_layout()
plt.show()

In [ ]:
def add_status_label(df):
    """Derive 3-class label from delay_minutes."""
    def label(d):
        if d < -1: return 'ahead'
        if d <= 2: return 'on_time'
        return 'behind'
    df = df.copy()
    df['status'] = df['delay_minutes'].apply(label)
    return df

train = add_status_label(train)
test  = add_status_label(test)

y_train = train['delay_minutes']
y_test  = test['delay_minutes']
y_train_cls = train['status']
y_test_cls  = test['status']

results = []  # We'll accumulate (model_name, mae, rmse, accuracy, f1) here

def evaluate(name, y_true_reg, y_pred_reg, y_true_cls, y_pred_cls):
    mae  = mean_absolute_error(y_true_reg, y_pred_reg)
    rmse = mean_squared_error(y_true_reg, y_pred_reg, squared=False)
    acc  = accuracy_score(y_true_cls, y_pred_cls)
    f1   = f1_score(y_true_cls, y_pred_cls, average='macro', labels=['ahead','on_time','behind'], zero_division=0)
    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'Accuracy': acc, 'Macro F1': f1})
    print(f'{name:30s}  MAE={mae:.3f}  RMSE={rmse:.3f}  Acc={acc:.3f}  F1={f1:.3f}')
    return mae, rmse

print(f'Test set class distribution:')
print(test['status'].value_counts(normalize=True).mul(100).round(1))

## Baseline 1: Always predict zero delay

This is the dumbest possible model. Its MAE tells us: *"if we just assumed every train was on time, how wrong would we be on average?"*

This is our absolute floor. Any real model must beat this.

In [ ]:
pred_zero = np.zeros(len(test))
cls_zero  = ['on_time'] * len(test)
evaluate('Zero (always on time)', y_test, pred_zero, y_test_cls, cls_zero)

## Baseline 2: Global mean

Predict the mean delay from the *training set* for every row. A slight improvement — at least it learns the overall bias (trains tend to run late or early on average).

Note: we train this on the training set only, then apply to the test set. This is the discipline we maintain for all models.

In [ ]:
global_mean = y_train.mean()
global_median = y_train.median()
print(f'Training set mean delay: {global_mean:.3f} min')
print(f'Training set median delay: {global_median:.3f} min')

def delay_to_status(d):
    if d < -1: return 'ahead'
    if d <= 2: return 'on_time'
    return 'behind'

pred_mean   = np.full(len(test), global_mean)
pred_median = np.full(len(test), global_median)
cls_mean    = [delay_to_status(global_mean)] * len(test)
cls_median  = [delay_to_status(global_median)] * len(test)

evaluate('Global mean', y_test, pred_mean, y_test_cls, cls_mean)
evaluate('Global median', y_test, pred_median, y_test_cls, cls_median)

## Baseline 3: Historical median by group

This is the most powerful simple baseline. For each combination of (route, station, hour-of-day, is_weekend), we compute the median delay from the training data and use that as our prediction.

**Why median, not mean?** Delay distributions are typically right-skewed (occasional very late trains pull the mean up). The median is more robust to outliers.

This is essentially a lookup table — no optimization, just historical facts. But it captures the core pattern: "Red Line at Belmont on a Monday morning is usually X minutes late."

In [ ]:
medians = (
    train.groupby(['route', 'station_id', 'hour', 'is_weekend'])['delay_minutes']
    .median()
    .reset_index()
    .rename(columns={'delay_minutes': 'median_delay'})
)

print(f'Lookup table has {len(medians):,} buckets')
print(f'Coverage: train has {train.groupby(["route","station_id","hour","is_weekend"]).ngroups} unique combos')

test_with_median = test.merge(
    medians, on=['route', 'station_id', 'hour', 'is_weekend'], how='left'
)
# Fall back to global median for unseen combos
test_with_median['median_delay'] = test_with_median['median_delay'].fillna(global_median)

pred_median_grp = test_with_median['median_delay'].values
cls_median_grp  = [delay_to_status(d) for d in pred_median_grp]

evaluate('Historical median (route×station×hour×weekend)', y_test, pred_median_grp, y_test_cls, cls_median_grp)

## Baseline 4: Ridge Regression

Ridge regression is a linear model with **L2 regularization** — it fits a line (hyperplane in high dimensions) to the data, but penalizes large coefficients to prevent overfitting.

**Why linear regression for a baseline?**
- Interpretable: you can look at each coefficient and understand what the model "thinks"
- Fast to train
- Surprisingly competitive when features are informative

**L2 regularization (Ridge):** adds a penalty term `α × Σ(coef²)` to the loss. This shrinks coefficients toward zero, which:
- Reduces overfitting
- Handles correlated features better than plain OLS
- `α` is the regularization strength — higher = more shrinkage

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    'is_red', 'is_blue', 'stop_sequence', 'direction',
    'hour', 'dow', 'is_weekend', 'is_peak_am', 'is_peak_pm',
    'minutes_until', 'is_scheduled', 'is_delayed', 'is_faulty',
    'eta_delta_1', 'eta_delta_2',
]

X_train = train[FEATURE_COLS].fillna(0)
X_test  = test[FEATURE_COLS].fillna(0)

# StandardScaler: transforms each feature to mean=0, std=1.
# This is REQUIRED for Ridge — without it, features with larger scales
# (like minutes_until which can be 0-60) dominate features like is_weekend (0/1).
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)
cls_ridge  = [delay_to_status(d) for d in pred_ridge]

evaluate('Ridge regression (α=1.0)', y_test, pred_ridge, y_test_cls, cls_ridge)

In [ ]:
# Inspect Ridge coefficients — what has the model learned?
coefs = pd.Series(
    ridge.named_steps['model'].coef_,
    index=FEATURE_COLS
).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
coefs.plot(kind='barh', ax=ax, color=coefs.apply(lambda x: 'tomato' if x > 0 else 'steelblue'))
ax.set_title('Ridge regression coefficients\n(positive = increases predicted delay)')
ax.axvline(0, color='white', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nInterpretation guide:")
print("  Large positive coef on 'is_delayed': model correctly learns that CTA's flag predicts delay")
print("  Positive coef on 'eta_delta_1': slipping ETA predicts lateness")
print("  Coefficient sign on 'is_peak_am'/'is_peak_pm': does rush hour increase or decrease delay?")

## Summary comparison

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results_df.plot(x='model', y='MAE', kind='bar', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('MAE (lower is better)')
axes[0].set_xlabel('')
axes[0].set_ylabel('Minutes')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

results_df.plot(x='model', y='Macro F1', kind='bar', ax=axes[1], legend=False, color='darkorange')
axes[1].set_title('Macro F1 (higher is better)')
axes[1].set_xlabel('')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

## Confusion matrix — what kinds of mistakes is the best model making?

A confusion matrix shows, for each true class, what the model predicted. Rows = truth, columns = prediction.

The key question: **Are we confusing "behind" with "on_time" (a soft mistake) or "ahead" with "behind" (a big mistake)?**

In [ ]:
labels = ['ahead', 'on_time', 'behind']
cm = confusion_matrix(y_test_cls, cls_ridge, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=ax
)
ax.set_title('Ridge: normalised confusion matrix')
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')
plt.tight_layout()
plt.show()

print("Off-diagonal cells = mistakes. High off-diagonal in 'on_time' column is common")
print("because most trains really are on time, so models default to predicting 'on_time'.")

## Key takeaways

1. **The zero baseline tells you the floor.** If your data has low variance in delay, even the zero model does OK.
2. **Historical median is a strong baseline** — it encodes domain knowledge (some stations at some times are reliably late) with no ML at all.
3. **Ridge adds value** by combining all features linearly. But it can't capture interactions (e.g., "Red Line + rush hour + is_delayed" is worse than any individual factor).
4. **The coefficients tell a story** — read them and check they make sense. If rush hour has a negative coefficient when you'd expect positive, something's wrong with your data or features.
5. **Confusion matrix matters** — pure accuracy hides whether you're making catastrophic errors.

➡️ **Next:** [04_ml_models.ipynb](04_ml_models.ipynb)